# 01 — Agents: baseline → hosted Groq

All default cells run without a key or model download. The last live cell is opt-in and consumes up to five Groq requests over roughly one minute. One teammate should run it after reserving the allowance in the shared quota tracker. Separate Colab clones do not share the backend SQLite budget.

In [ ]:
import sys, subprocess, tempfile
from pathlib import Path

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "backend/app/world.py").exists()), None)
if ROOT is None and "google.colab" in sys.modules:
    ROOT = Path("/content/MiniDavis")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "https://github.com/KTH-Sys/MiniDavis.git", str(ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "backend/requirements.txt")], check=True)
if ROOT is None:
    raise RuntimeError("Open this notebook from a MiniDavis checkout.")
sys.path.insert(0, str(ROOT))
print("Project imports ready.")


In [ ]:
from backend.app.agents import baseline_plan
from backend.app.world import World
from backend.app.validation import validate_model_action, InvalidModelAction
from backend.app.schemas import StepRequest
world = World(seed=11)
state = world.snapshot()
for resident in state.agents:
    action = baseline_plan(resident, state)
    envelope = {"run_id": state.run_id, "agent_id": resident.id, "observation_revision": state.revision, **action.model_dump(exclude_none=True)}
    assert validate_model_action(envelope, resident, state) == action
    print(resident.name, action.model_dump())


## Observe and remember
The next cell executes a turn and inspects the exact previous source events supplied before the following decision.

In [ ]:
for turn in range(2):
    state = world.snapshot()
    state = world.step(StepRequest(run_id=state.run_id, expected_revision=state.revision, request_id=f"notebook-{turn}"))
inspector = world.inspect("ethan")
assert inspector.decision.memory.recent
for event in inspector.decision.memory.recent:
    print(event.event_id, event.text)


## Reject an invalid proposal
The model proposes actions; world validation still controls execution.

In [ ]:
resident = state.agents[0]
try:
    validate_model_action({"run_id": state.run_id, "agent_id": resident.id, "observation_revision": state.revision, "action": "fly"}, resident, state)
except InvalidModelAction:
    print("Invalid action rejected safely.")
else:
    raise AssertionError("Invalid action was accepted")


## Connection setup check — no provider call
Reports only whether a server key exists. It never prints credentials.

In [ ]:
subprocess.run([sys.executable, str(ROOT / "scripts/check_groq.py")], cwd=ROOT, check=True)


## Optional live verification
Set `RUN_LIVE = True` only after confirming Qwen access and reserving five requests. In Colab, create a Secret named `LLM_API_KEY` and allow this notebook to access it. Locally, use `backend/.env`. The production adapter handles strict JSON, four requests/minute, one in-flight request, cached routes, timeouts, Retry-After and stale-response rejection. This check pauses its town while waiting and saves actual executed evidence. A 401/403 means check access; a 429 means wait for quota. Do not share the key or run this cell concurrently across teammates.

In [ ]:
RUN_LIVE = False
if RUN_LIVE:
    import os
    if "google.colab" in sys.modules:
        from google.colab import userdata
        os.environ["LLM_API_KEY"] = userdata.get("LLM_API_KEY")
    subprocess.run([sys.executable, str(ROOT / "scripts/check_groq.py"), "--live"], cwd=ROOT, check=True)
else:
    print("Live verification skipped; no Groq request made.")


## Failure checks without Groq
Run `python -m pytest backend/tests/test_groq.py -q` from the repository root. These tests use fake provider replies and cover budgets, missing keys, invalid actions, timeouts, 429 cooldowns, cached routes and resets. Do not call mocked results live evidence.